In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

print("Testing Phase 1 Model...\n")

LORA_PATH = 'qwen_medical_lora_gpu'
BASE_PATH = 'Qwen25-7B'

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA - use absolute path without validation
print("Loading Phase 1 LoRA...")
model = PeftModel.from_pretrained(
    base_model, 
    LORA_PATH,
    is_trainable=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✓ Model loaded\n")

# Test inference
print("="*80)
print("INFERENCE TEST")
print("="*80)

test_prompts = [
    "Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N2->N2,RP at N3->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required.",
    "Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1 . Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required.",
    "Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N1->N2,RP at N2->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required."
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=600, temperature=0.5)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {result}...\n")

print("="*80)
print("✓ Phase 1 model is working!")
print("="*80)

Testing Phase 1 Model...

Loading base model...


Loading weights: 100%|███████████████████████████████████████████████████████████████| 339/339 [00:02<00:00, 169.12it/s]


Loading Phase 1 LoRA...
✓ Model loaded

INFERENCE TEST

Q: Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N2->N2,RP at N3->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required.
A: Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N2->N2,RP at N3->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required. Shunt type: 3
Ligation strategy: RP at N3->N1

`

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

print("Testing Phase 1 Model...\n")

LORA_PATH = 'qwen_medical_lora_gpu'
BASE_PATH = 'Qwen25-7B'

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA - use absolute path without validation
print("Loading Phase 1 LoRA...")
model = PeftModel.from_pretrained(
    base_model, 
    LORA_PATH,
    is_trainable=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✓ Model loaded\n")

In [24]:
chiva_rules = """
=== CHIVA VENOUS SHUNT CLASSIFICATION RULES ===

ANATOMY:
    N1 = Deep venous system (femoral / popliteal vein)
    N2 = Great Saphenous Vein (GSV) or Small Saphenous Vein (SSV) trunk
    N3 = Tributaries / superficial branches
    EP = Physiological (forward, antegrade) flow — NORMAL clip
    RP = Retrograde (pathological, reflux) flow — ABNORMAL clip
    SFJ = Saphenofemoral Junction  →  posYRatio ≤ 0.098
    Hunterian Perforator            →  0.098 < posYRatio ≤ 0.353

═══════════════════════════════════════════════════════════
CRITICAL RULE — SFJ COMPETENCE (read before classifying):
    SFJ is INCOMPETENT if and only if a clip has fromType=N1 AND toType=N2 (EP N1→N2).
    EP N2→N2 means blood circulates within the saphenous trunk via a perforator — SFJ REMAINS COMPETENT.
    This is true regardless of posYRatio or step label. Even posYRatio=0.05 with step=SFJ-Knee
    is a perforator entry if the clip reads EP N2→N2, NOT EP N1→N2.
═══════════════════════════════════════════════════════════

STEP 1 — CHECK FOR EP N1→N2:
    Scan ALL clips. Does any clip have flow=EP, fromType=N1, toType=N2?
    YES → SFJ/Hunterian INCOMPETENT → go to Case A or B.
    NO  → SFJ COMPETENT → go to Case C.

─────────────────────────────────────────────────────────
Case A — EP N1→N2 EXISTS (SFJ or Hunterian), NO EP N2→N3
─────────────────────────────────────────────────────────
    If RP N2→N1 present AND no RP at N3 (no RP N3→N2, no RP N3→N1) → TYPE 1
    Ligation: Ligate at SFJ (y≤0.098) or Hunterian (y≤0.353).
            If multiple RP N2→N1: ligate below each except the most distal.

─────────────────────────────────────────────────────────
Case B — EP N1→N2 EXISTS (SFJ or Hunterian) AND EP N2→N3 EXISTS
─────────────────────────────────────────────────────────
    B1: RP N3→N2 or RP N3→N1, NO RP N2→N1               → TYPE 3
    B2: RP N3→N2 AND RP N2→N1                             → TYPE 3
    B3: RP N3→N1 AND RP N2→N1, eliminationTest absent    → UNDETERMINED (set needs_elim_test=true)
    B4: RP N3→N1 AND RP N2→N1, eliminationTest="Reflux"  → TYPE 1+2
    B5: RP N3→N1 AND RP N2→N1, eliminationTest="No Reflux" → TYPE 3

    TYPE 3 Ligation:
        Single RP at N3: Ligate EP at N2→N3. Follow up 6–12 months; if N2 reflux develops, ligate SFJ.
        Multiple RP at N3: Ligate every refluxing tributary at N2 junction (CHIVA 2 step 1). Same follow-up.

    TYPE 1+2 Ligation — depends on RP N2→N1 calibre:
        Small RP N2→N1: Apply CHIVA 2 (ligate EP N2→N3 first, then SFJ/Hunterian).
                        OR ligate SFJ first + all tributaries except one; once N2 normalises ligate last.
        Large / multiple RP N2→N1: Ligate SFJ/Hunterian + every refluxing tributary simultaneously.
                                    Ligate below each RP N2→N1 except the most distal.

─────────────────────────────────────────────────────────
Case C — NO EP N1→N2 ANYWHERE (SFJ COMPETENT)
─────────────────────────────────────────────────────────
    C-Sub-check: what type of EP clip exists?

    ── TYPE 2A ── EP N2→N3 present, NO EP N1→N2
        The defining feature is EP N2→N3 (GSV feeding a tributary) without any SFJ entry.
        RP may or may not be present in early/developing cases.
        Typical pattern: EP N2→N3 + RP N3→N2 or N3→N1. No RP N2→N1.
        Key signal: EP N2→N3 clip exists + NO EP N1→N2 clip exists anywhere.
        If multiple RP at N3 → set ask_branching=true (need calibre/distance/drainage info).
        Ligation: Ligate highest EP at N2→N3 junction.
                    If multiple branching at N3: ligate based on calibre, distance to perforator, drainage.

    ── TYPE 2B ── EP N2→N2 present, NO EP N1→N2, RP at N3, NO RP N2→N1
        Entry is via perforator (fromType=N2, toType=N2 — NOT N1→N2).
        IMPORTANT: EP N2→N2 at ANY posYRatio (even 0.05, SFJ-Knee step) = perforator, NOT SFJ.
        Key signal: EP N2→N2 clip + RP N3→N2 or N3→N1 + NO EP N1→N2 + NO RP N2→N1.
        If multiple RP at N3 → set ask_branching=true.
        Ligation: Ligate the highest EP N2→N2 (perforator entry point).

    ── TYPE 2C ── EP N2→N2 present, NO EP N1→N2, RP at N3, RP N2→N1 ALSO present
        Perforator entry (EP N2→N2) with secondary GSV reflux (RP N2→N1). SFJ still competent.
        IMPORTANT: 2C has EP N2→N2 (perforator), while Type 1+2 has EP N1→N2 (SFJ entry).
        If NO EP N1→N2 but RP N2→N1 exists with EP N2→N2 → TYPE 2C, not Type 1+2.
        Key signal: EP N2→N2 + RP N3 + RP N2→N1 + NO EP N1→N2.
        Ligation: Ligate perforator entry (highest EP N2→N2) AND all RP N2→N1 sites along GSV.

    Case C — NO SHUNT:
        If EP N2→N2 exists but NO RP clips of any kind → NO SHUNT DETECTED.

─────────────────────────────────────────────────────────
Case D — No RP in any clip → NO SHUNT DETECTED. No ligation needed.
─────────────────────────────────────────────────────────

QUICK DECISION TABLE (commit this to memory):
    Has EP N1→N2? YES + no EP N2→N3 + RP N2→N1           → TYPE 1
    Has EP N1→N2? YES + EP N2→N3 + RP N3 only             → TYPE 3
    Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + eliminationTest absent → UNDETERMINED
    Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + elim="Reflux"          → TYPE 1+2
    Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + elim="No Reflux"       → TYPE 3
    No EP N1→N2  + EP N2→N3                                → TYPE 2A
    No EP N1→N2  + EP N2→N2 + RP N3 + NO RP N2→N1         → TYPE 2B
    No EP N1→N2  + EP N2→N2 + RP N3 + RP N2→N1            → TYPE 2C
    No EP N1→N2  + EP N2→N2 + NO RP                        → NO SHUNT
    EP N1→N3 + RP N2→N1                                    → TYPE 4
    EP N1→N3 + RP N3→N2 or RP N3→N1                         → TYPE 5
    No RP at all                                            → NO SHUNT

CONCRETE EXAMPLES (match these patterns exactly):
    Type 1:  [EP N1→N2 y=0.06 SFJ-ENTRY, RP N2→N1 y=0.25]
            → EP N1→N2 present, RP N2→N1, no EP N2→N3, no N3 reflux → TYPE 1
    Type 2A: [EP N2→N3 y=0.20]  OR  [EP N2→N3 y=0.20, RP N3→N2 y=0.47]
            → No EP N1→N2, EP N2→N3 present → TYPE 2A
    Type 2B: [EP N2→N2 y=0.050 step=SFJ-Knee ligation-point-marker, RP N3→N1 y=0.132]
            → No EP N1→N2, EP N2→N2 = perforator, RP N3 only → TYPE 2B
    Type 2C: [EP N2→N2 y=0.050 step=SFJ-Knee ligation-point-marker, RP N3→N1 y=0.132, RP N2→N1 y=0.212]
            → No EP N1→N2, EP N2→N2 = perforator, RP N3 + RP N2→N1 → TYPE 2C
    Type 3:  [EP N1→N2 y=0.05 SFJ-ENTRY, EP N2→N3 y=0.132 ligation-point-marker, RP N3→N1 y=0.212]
            → EP N1→N2 + EP N2→N3 + RP N3→N1, no RP N2→N1 → TYPE 3
        Type 4:  [EP N1→N3 y=0.60, RP N2→N1 y=0.40]
            → EP N1→N3 with N2 return → TYPE 4
        Type 5:  [EP N1→N3 y=0.65, RP N3→N2 y=0.50, RP N3→N1 y=0.75]
            → EP N1→N3 with looping N3 return → TYPE 5
    Type 3 variant 2 (no elim test):
            [EP N1→N2, EP N2→N3, RP N3→N1, RP N2→N1, no eliminationTest] → UNDETERMINED
    Type 1+2:[EP N1→N2, EP N2→N3 eliminationTest="Reflux", RP N3→N1, RP N2→N1] → TYPE 1+2
    No shunt:[EP N1→N2 only, no RP]  OR  [EP N2→N2 only, no RP] → NO SHUNT

TYPE 2 BRANCHING — ask_branching flag:
    Set ask_branching=true when there are MULTIPLE RP at N3 tributaries in a Type 2A, 2B, or 2C case.
    The ligation choice among multiple N3 branches depends on:
        • Calibre of branches (equal or unequal)
        • Distance of each branch to its perforator
        • Whether drainage through the thinner vessel is possible
    If unequal calibre with drainage possible → ligate the larger vessel.
    If unequal calibre, no drainage → ligate the smaller vessel.
    If equal calibre, unequal distance → ligate the branch with longer distance to perforator.

COORDINATE HINTS (secondary — always check fromType/toType first):
    posYRatio ≤ 0.098   = SFJ region (upper thigh)
    0.099–0.353         = Hunterian / mid-thigh
    0.354–0.60          = Knee / popliteal
    > 0.60              = Calf / ankle (SPJ region for posterior clips)

OUTPUT FLAGS:
    needs_elim_test : true when RP N3→N1 + RP N2→N1 present but eliminationTest is absent (B3)
    ask_branching   : true for Type 2A/2B/2C with multiple RP at N3

CONFIDENCE GUIDE:
    Clear single pattern, no ambiguity         → 0.90–0.97
    Pattern present but some noise clips       → 0.80–0.89
    Ambiguous (needs elimination test)         → 0.50–0.65
    No pattern / insufficient clips            → 0.40–0.55
"""

base_prompt = """
The input clip data is : {input_clip}

═══════════════════════════════════════════════════════════════
STEP-BY-STEP DECISION GUIDE (Follow in order)
═══════════════════════════════════════════════════════════════

STEP 1: CHECK FOR EP N1→N2 (SFJ or Hunterian ENTRY)
    Look for: "EP N1→N2" with y≤0.098 (SFJ) or y≤0.353 (Hunterian)
    If YES with SFJ-ENTRY/Hunterian-ENTRY label → SFJ INCOMPETENT
    If NO  → SFJ COMPETENT (go to Case C)
    ✓ Found EP N1→N2? YES/NO

    STEP 2: IF YES to EP N1→N2, CHECK FOR REFLUX PATTERNS
    2a) ANY RP N3→N2 or RP N3→N1? (tributary reflux)
    2b) ANY RP N2→N1? (GSV reflux)
    2c) ANY RP anywhere else?
    2d) ANY EP N2→N3? (extra antegrade to tributary)

STEP 3: MATCH PATTERN TO TYPE

    ┌─ SFJ INCOMPETENT PATH (has EP N1→N2):
    │
    ├─ NO EP N2→N3:
    │  └─ Has RP N2→N1, no RP at N3 → TYPE 1 (confidence 0.90)
    │
    └─ YES EP N2→N3 EXISTS:
        ├─ Has RP N3 (at N2 or N1), NO RP N2→N1 → TYPE 3 (confidence 0.88)
        ├─ Has RP N3 AND RP N2→N1:
        │  ├─ eliminationTest absent → UNDETERMINED (confidence 0.55) [needs_elim_test=true]
        │  ├─ eliminationTest="Reflux" → TYPE 1+2 (confidence 0.80) 
        │  └─ eliminationTest="No Reflux" → TYPE 3 (confidence 0.75)

    ┌─ SFJ COMPETENT PATH (NO EP N1→N2):
    │
    ├─ EP N2→N3 EXISTS:
    │  └─ TYPE 2A (confidence 0.85-0.92)
    │     └─ Multiple RP at N3? → [ask_branching=true]
    │
    └─ ONLY EP N2→N2 (perforator entry):
        ├─ Has RP N3, NO RP N2→N1 → TYPE 2B (confidence 0.84)
        │  └─ Multiple RP at N3? → [ask_branching=true]
        ├─ Has RP N3 AND RP N2→N1 → TYPE 2C (confidence 0.82)
        │  └─ Multiple RP at N3? → [ask_branching=true]
        └─ No RP at all → NO SHUNT (confidence 0.95)

STEP 4: ASSIGN CONFIDENCE
    Clear pattern, no ambiguity → 0.90–0.97
    Pattern present, minor noise → 0.80–0.89
    Ambiguous / needs elimination test → 0.50–0.65
    Insufficient clips → 0.40–0.55

═══════════════════════════════════════════════════════════════
CRITICAL REMINDERS:
    • EP N1→N2 is THE KEY decision point — check this FIRST
    • EP N2→N2 means perforator (SFJ COMPETENT), never confuse with N1→N2
    • Type 2A has EP N2→N3; Type 2B/2C have EP N2→N2 (NOT N2→N3)
    • Type 2C differs from Type 1+2: 2C has EP N2→N2, Type 1+2 has EP N1→N2
    • Type 4/5 are N1→N3 path shunts and should be classified explicitly when present
    • RP only at N3 (not N2→N1) + EP N1→N2 = TYPE 3 (not 1+2)
═══════════════════════════════════════════════════════════════

The chiva rules are : {chiva_rules}

=== SHUNT CLASSIFICATION TASK ===
Follow the Step-by-Step Decision Guide above.
Output ONLY the JSON below — no other text, no markdown.

=== LIGATION TASK ===
Based on the shunt type you identified in previous task, the clinical findings above, and the medical knowledge base provided:

1. Generate a detailed ligation plan with specific steps
2. Identify any additional clinical information needed
3. Consider complications and contraindications
4. Provide follow-up and monitoring recommendations
5. Consider CHIVA principles (hemodynamic, saphenous-vein-sparing when appropriate)

Important formatting rules:
1. ligation_steps must be a JSON array with one clear action per item.
2. Each ligation step must name the ligation point or vessel segment explicitly.
3. clinical_rationale must explain why that plan fits the shunt anatomy.
4. additional_info_needed must be [] when there is no meaningful extra information to request.
5. chiva_approach must describe the hemodynamic CHIVA reasoning, even if brief.


{{{{
    "shunt_type": "<Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 4 / Type 5 / Type 1+2 / No shunt detected / Undetermined>",
    "confidence": <0.0-1.0>,
    "reasoning": ["<decision step 1>", "<decision step 2>", "..."],
    "ligation planning" : "<3 point summary>"
    "ask_branching": <true/false>,
    "summary": "<1 sentence clinical summary>"
}}}}"""

In [25]:
test_clips[0]

'EP at N2->N2,RP at N3->N1.'

In [26]:
prompt = base_prompt.format(input_clip=test_clips[0],chiva_rules=chiva_rules)
print(prompt)


The input clip data is : EP at N2->N2,RP at N3->N1.

═══════════════════════════════════════════════════════════════
STEP-BY-STEP DECISION GUIDE (Follow in order)
═══════════════════════════════════════════════════════════════

STEP 1: CHECK FOR EP N1→N2 (SFJ or Hunterian ENTRY)
    Look for: "EP N1→N2" with y≤0.098 (SFJ) or y≤0.353 (Hunterian)
    If YES with SFJ-ENTRY/Hunterian-ENTRY label → SFJ INCOMPETENT
    If NO  → SFJ COMPETENT (go to Case C)
    ✓ Found EP N1→N2? YES/NO

    STEP 2: IF YES to EP N1→N2, CHECK FOR REFLUX PATTERNS
    2a) ANY RP N3→N2 or RP N3→N1? (tributary reflux)
    2b) ANY RP N2→N1? (GSV reflux)
    2c) ANY RP anywhere else?
    2d) ANY EP N2→N3? (extra antegrade to tributary)

STEP 3: MATCH PATTERN TO TYPE

    ┌─ SFJ INCOMPETENT PATH (has EP N1→N2):
    │
    ├─ NO EP N2→N3:
    │  └─ Has RP N2→N1, no RP at N3 → TYPE 1 (confidence 0.90)
    │
    └─ YES EP N2→N3 EXISTS:
        ├─ Has RP N3 (at N2 or N1), NO RP N2→N1 → TYPE 3 (confidence 0.88)
        ├─ 

In [29]:
test_clips = [
    "EP at N2->N2,RP at N3->N1.",
    "EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.",
    "EP at N1->N2,RP at N2->N1."
]

for clip in test_clips:
    prompt = base_prompt.format(input_clip=clip,chiva_rules=chiva_rules)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=6000, temperature=0.5)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("++++++++++++++++++++++++START OF NEW RESPONSE++++++++++++++++++++++++++++++++")
    print(f"A: {result}...\n")

++++++++++++++++++++++++START OF NEW RESPONSE++++++++++++++++++++++++++++++++
A: 
The input clip data is : EP at N2->N2,RP at N3->N1.

═══════════════════════════════════════════════════════════════
STEP-BY-STEP DECISION GUIDE (Follow in order)
═══════════════════════════════════════════════════════════════

STEP 1: CHECK FOR EP N1→N2 (SFJ or Hunterian ENTRY)
    Look for: "EP N1→N2" with y≤0.098 (SFJ) or y≤0.353 (Hunterian)
    If YES with SFJ-ENTRY/Hunterian-ENTRY label → SFJ INCOMPETENT
    If NO  → SFJ COMPETENT (go to Case C)
    ✓ Found EP N1→N2? YES/NO

    STEP 2: IF YES to EP N1→N2, CHECK FOR REFLUX PATTERNS
    2a) ANY RP N3→N2 or RP N3→N1? (tributary reflux)
    2b) ANY RP N2→N1? (GSV reflux)
    2c) ANY RP anywhere else?
    2d) ANY EP N2→N3? (extra antegrade to tributary)

STEP 3: MATCH PATTERN TO TYPE

    ┌─ SFJ INCOMPETENT PATH (has EP N1→N2):
    │
    ├─ NO EP N2→N3:
    │  └─ Has RP N2→N1, no RP at N3 → TYPE 1 (confidence 0.90)
    │
    └─ YES EP N2→N3 EXISTS:
     

In [33]:
import json

def extract_json_from_response(text):
    """Extract JSON from model output with ```json markers."""

    # Look for ```json marker
    json_marker = "```json"
    idx = text.find(json_marker)

    if idx != -1:
        # Found ```json, skip past the marker and any newlines
        start_search = idx + len(json_marker)
        text = text[start_search:]

    # Find first { and match it with closing }
    start_idx = text.find('{')
    if start_idx == -1:
        return None

    brace_count = 0
    end_idx = start_idx

    for i in range(start_idx, len(text)):
        if text[i] == '{':
            brace_count += 1
        elif text[i] == '}':
            brace_count -= 1
            if brace_count == 0:
                end_idx = i + 1
                break

    if end_idx == start_idx:
        return None

    candidate = text[start_idx:end_idx].strip()

    # Try to parse as JSON
    try:
        parsed = json.loads(candidate)
        return parsed
    except json.JSONDecodeError as e:
        print(f"❌ JSON parse error: {e}")
        print(f"Snippet: {candidate[:150]}...")
        return None

def clean_model_output(text):
    """Extract and pretty-print the JSON from model output."""
    result = extract_json_from_response(text)

    if result:
        print("✓ Successfully extracted JSON:")
        print(json.dumps(result, indent=2))
        return result
    else:
        print("✗ Could not extract valid JSON from model output")
        print("\nRaw output (first 500 chars):")
        print(text[:500])
        return None


# Example usage:
if __name__ == "__main__":
    # Simulate messy model output
    messy_output = """
    <|im_start|>user
    The input clip data is : EP at N2->N2,RP at N3->N1.
    ...
    [TONS OF PROMPT TEXT HERE]
    ...
    <|im_start|>assistant
    {
        "shunt_type": "Type 2B",
        "confidence": 0.88,
        "reasoning": ["No EP N1->N2", "EP N2->N2 present", "RP at N3->N1"],
        "summary": "Perforator-fed shunt with tributary reflux"
    }
    """

    result = clean_model_output(messy_output)

✓ Successfully extracted JSON:
{
  "shunt_type": "Type 2B",
  "confidence": 0.88,
  "reasoning": [
    "No EP N1->N2",
    "EP N2->N2 present",
    "RP at N3->N1"
  ],
  "summary": "Perforator-fed shunt with tributary reflux"
}


In [34]:
test_clips = [
    "EP at N2->N2,RP at N3->N1.",
    "EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.",
    "EP at N1->N2,RP at N2->N1."
]

for clip in test_clips:
    prompt = base_prompt.replace("{input_clip}", clip).replace("{chiva_rules}", chiva_rules)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=6000, temperature=0.5)
    
    raw_result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Clip: {clip}")
    print(f"{'='*60}")
    
    # Extract clean JSON
    classification = clean_model_output(raw_result)
    if classification:
        print(f"Type: {classification.get('shunt_type')}")
        print(f"Confidence: {classification.get('confidence')}")


Clip: EP at N2->N2,RP at N3->N1.
✓ Successfully extracted JSON:
{
  "shunt_type": "Type 1",
  "confidence": 0.95,
  "reasoning": [
    "Has EP N1\u2192N2? YES",
    "Has EP N2\u2192N3? NO",
    "Has RP N3\u2192N2 or N3\u2192N1? NO",
    "Has RP N2\u2192N1? NO"
  ],
  "ligation_planning": {
    "steps": [
      "Ligate the SFJ at the knee.",
      "Follow up in 6-12 months to assess N2 normalization."
    ]
  },
  "clinical_rationale": "The presence of an EP N1\u2192N2 indicates SFJ incompetence. The absence of RP N2\u2192N3 and N3 reflux suggests a simple Type 1 shunt with no additional reflux points. Ligation of the SFJ will address the primary reflux source. Follow-up is recommended to ensure proper healing and to monitor for any new reflux development.",
  "additional_info_needed": [],
  "chiva_approach": "This is a typical CHIVA procedure as it involves ligation of the SFJ, which is the primary site of reflux. The saphenous vein will be preserved to maintain venous return and prev

In [36]:
qwen_optimized_prompt = """CHIVA VENOUS SHUNT CLASSIFICATION — QWEN 7B OPTIMIZED

CRITICAL DECISION RULE:
EP N1→N2 (SFJ/Hunterian entry, fromType=N1, toType=N2) → SFJ INCOMPETENT
EP N2→N2 (perforator entry, fromType=N2, toType=N2) → SFJ COMPETENT (regardless of posYRatio)

STEP 1: Does the input contain EP N1→N2?

YES → SFJ INCOMPETENT PATH
  Check: Is there EP N2→N3?

  NO EP N2→N3:
    RP N2→N1 present, no RP N3 → TYPE 1

  YES EP N2→N3:
    Only RP N3 (no RP N2→N1) → TYPE 3
    RP N3 AND RP N2→N1, eliminationTest absent → UNDETERMINED (needs_elim_test=true)
    RP N3 AND RP N2→N1, eliminationTest="Reflux" → TYPE 1+2
    RP N3 AND RP N2→N1, eliminationTest="No Reflux" → TYPE 3

NO → SFJ COMPETENT PATH (no EP N1→N2 anywhere)
  Check: What EP clip exists?

  EP N2→N3 present → TYPE 2A
  EP N2→N2 (perforator) + RP N3 only (no RP N2→N1) → TYPE 2B
  EP N2→N2 (perforator) + RP N3 + RP N2→N1 → TYPE 2C
  EP N2→N2 + NO RP → NO SHUNT
  No EP clips or no RP anywhere → NO SHUNT

LIGATION RULES:
  TYPE 1: Ligate SFJ (y≤0.098) or Hunterian (y≤0.353)
  TYPE 2A: Ligate highest EP at N2→N3 junction
  TYPE 2B: Ligate highest EP N2→N2 (perforator entry)
  TYPE 2C: Ligate EP N2→N2 + all RP N2→N1 sites
  TYPE 3: Ligate EP N2→N3; follow-up 6-12 months
  TYPE 1+2: Small RP N2→N1: CHIVA 2 (ligate EP N2→N3 first, then SFJ). Large: ligate SFJ + tributaries

EXAMPLES (exact patterns):
  TYPE 1: [EP N1→N2, RP N2→N1] → SFJ incompetent, no extra paths
  TYPE 2A: [EP N2→N3, RP N3→N2 or RP N3→N1] → No SFJ entry, GSV feeds tributary
  TYPE 2B: [EP N2→N2 (y=0.05), RP N3→N1] → Perforator entry, tributary reflux only
  TYPE 2C: [EP N2→N2, RP N3→N1, RP N2→N1] → Perforator + secondary GSV reflux
  TYPE 3: [EP N1→N2, EP N2→N3, RP N3→N1] → SFJ incompetent but RP only at N3

═══════════════════════════════════════════════════════════════
INPUT CLIP DATA: {input}
═══════════════════════════════════════════════════════════════

CLASSIFY THIS SHUNT NOW.

OUTPUT ONLY VALID JSON — NO OTHER TEXT, NO MARKDOWN, NO CODE BLOCKS.

{{
    "shunt_type": "<Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No shunt detected / Undetermined>",
    "confidence": <0.0-1.0>,
    "reasoning": ["<step 1>", "<step 2>", "<step 3>"],
    "ligation_steps": ["<specific point 1>", "<specific point 2>"],
    "ask_branching": <true/false>,
    "summary": "<1 sentence>"
}}"""

# Usage:
if __name__ == "__main__":
    test_clips = [
        "EP at N2->N2,RP at N3->N1.",
        "EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.",
        "EP at N1->N2,RP at N2->N1."
    ]

    for clip in test_clips:
        prompt = qwen_optimized_prompt.format(input=clip)
        print(f"Prompt tokens (approx): {len(prompt.split()) // 0.75}")  # Rough estimate
        print(f"\n{'='*60}\nClip: {clip}\n{'='*60}\n")

        # TODO: Call Qwen 7B API here
        # outputs = qwen_model.generate(prompt, max_new_tokens=800)

Prompt tokens (approx): 501.0

Clip: EP at N2->N2,RP at N3->N1.

Prompt tokens (approx): 509.0

Clip: EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.

Prompt tokens (approx): 501.0

Clip: EP at N1->N2,RP at N2->N1.



In [41]:
import torch

test_clips = [
    "EP at N2->N2,RP at N3->N1.",
    "EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.",
    "EP at N1->N2,RP at N2->N1."
]

for clip in test_clips:
    prompt = qwen_optimized_prompt.format(input=clip)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=6000,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2  # ← Prevents echoing
        )
        print("Outputs:")
        print(outputs)
    
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = clean_model_output(raw_output)
    
    print(f"\nClip: {clip}")
    print(f"Type: {result['shunt_type']}")
    print(f"Confidence: {result['confidence']}")
    print(f"Ligation: {result['ligation_steps']}")

Outputs:
tensor([[  2149,  91340,    647,  ...,   2548,      0, 151643]],
       device='cuda:0')
❌ JSON parse error: Expecting value: line 3 column 19 (char 135)
Snippet: {
    "shunt_type": "<Type 1 / Type 2A / Type 2B / Type 2C / Type 3 / Type 1+2 / No shunt detected / Undetermined>",
    "confidence": <0.0-1.0>,
    ...
✗ Could not extract valid JSON from model output

Raw output (first 500 chars):
CHIVA VENOUS SHUNT CLASSIFICATION — QWEN 7B OPTIMIZED

CRITICAL DECISION RULE:
EP N1→N2 (SFJ/Hunterian entry, fromType=N1, toType=N2) → SFJ INCOMPETENT
EP N2→N2 (perforator entry, fromType=N2, toType=N2) → SFJ COMPETENT (regardless of posYRatio)

STEP 1: Does the input contain EP N1→N2?

YES → SFJ INCOMPETENT PATH
  Check: Is there EP N2→N3?

  NO EP N2→N3:
    RP N2→N1 present, no RP N3 → TYPE 1

  YES EP N2→N3:
    Only RP N3 (no RP N2→N1) → TYPE 3
    RP N3 AND RP N2→N1, eliminationTest absen

Clip: EP at N2->N2,RP at N3->N1.


TypeError: 'NoneType' object is not subscriptable

In [42]:
qwen_minimal_prompt = """TASK: Classify CHIVA venous shunt. Return ONLY JSON. Do NOT repeat the input. Do NOT output any text except JSON.

General Notes:
EP N1→N2 = SFJ incompetent | EP N2→N2 = SFJ competent (perforator)

Quick Reference Table: 
TYPE 1: EP N1→N2, RP N2→N1, no EP N2→N3, no RP N3
TYPE 2A: EP N2→N3, no EP N1→N2
TYPE 2B: EP N2→N2, RP N3, no RP N2→N1, no EP N1→N2
TYPE 2C: EP N2→N2, RP N3, RP N2→N1, no EP N1→N2
TYPE 3: EP N1→N2, EP N2→N3, RP N3 only
TYPE 1+2: EP N1→N2, EP N2→N3, RP N3 + RP N2→N1, elim test="Reflux"
UNDETERMINED: EP N1→N2, EP N2→N3, RP N3 + RP N2→N1, no elim test
NO SHUNT: No RP anywhere

CLIP: {input}

RESPOND WITH ONLY THIS JSON FORMAT. NO OTHER TEXT:
{{"shunt_type": "TYPE", "confidence": 0.9, "reasoning": ["r1", "r2"], "ligation_steps": ["s1", "s2"], "summary": "text"}}"""

if __name__ == "__main__":
    test_clips = [
        "EP at N2->N2,RP at N3->N1.",
        "EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.",
        "EP at N1->N2,RP at N2->N1."
    ]

    for clip in test_clips:
        prompt = qwen_minimal_prompt.format(input=clip)
        print(f"Prompt size: {len(prompt)} chars, ~{len(prompt.split())} words")
        print(f"\n{'='*60}\nClip: {clip}\n{'='*60}")
        print(f"Prompt:\n{prompt}\n")

Prompt size: 803 chars, ~146 words

Clip: EP at N2->N2,RP at N3->N1.
Prompt:
TASK: Classify CHIVA venous shunt. Return ONLY JSON. Do NOT repeat the input. Do NOT output any text except JSON.

General Notes:
EP N1→N2 = SFJ incompetent | EP N2→N2 = SFJ competent (perforator)

Quick Reference Table: 
TYPE 1: EP N1→N2, RP N2→N1, no EP N2→N3, no RP N3
TYPE 2A: EP N2→N3, no EP N1→N2
TYPE 2B: EP N2→N2, RP N3, no RP N2→N1, no EP N1→N2
TYPE 2C: EP N2→N2, RP N3, RP N2→N1, no EP N1→N2
TYPE 3: EP N1→N2, EP N2→N3, RP N3 only
TYPE 1+2: EP N1→N2, EP N2→N3, RP N3 + RP N2→N1, elim test="Reflux"
UNDETERMINED: EP N1→N2, EP N2→N3, RP N3 + RP N2→N1, no elim test
NO SHUNT: No RP anywhere

CLIP: EP at N2->N2,RP at N3->N1.

RESPOND WITH ONLY THIS JSON FORMAT. NO OTHER TEXT:
{"shunt_type": "TYPE", "confidence": 0.9, "reasoning": ["r1", "r2"], "ligation_steps": ["s1", "s2"], "summary": "text"}

Prompt size: 831 chars, ~152 words

Clip: EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.
Prompt:
TASK: Classif

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

print("Testing Phase 1 Model...\n")

LORA_PATH = 'qwen_medical_lora_gpu'
BASE_PATH = 'Qwen25-7B'

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA
print("Loading Phase 1 LoRA...")
model = PeftModel.from_pretrained(
    base_model, 
    LORA_PATH,
    is_trainable=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✓ Model loaded\n")

# TESTING CODE
prompt_template = """Classify this CHIVA venous shunt.

INPUT: {input}

TYPES:
TYPE 1: EP N1→N2, RP N2→N1, no EP N2→N3, no RP N3
TYPE 2A: EP N2→N3, no EP N1→N2
TYPE 2B: EP N2→N2, RP N3, no RP N2→N1, no EP N1→N2
TYPE 2C: EP N2→N2, RP N3, RP N2→N1, no EP N1→N2
TYPE 3: EP N1→N2, EP N2→N3, RP N3 only

Output:"""

test_clips = [
    "EP at N2->N2,RP at N3->N1.",
    "EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.",
    "EP at N1->N2,RP at N2->N1."
]

for clip in test_clips:
    print(f"\n{'='*80}")
    print(f"INPUT: {clip}")
    print(f"{'='*80}\n")
    
    prompt = prompt_template.format(input=clip)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("RAW OUTPUT:")
    print(raw_output)
    print("\n")

Testing Phase 1 Model...

Loading base model...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 339/339 [00:04<00:00, 79.35it/s]


Loading Phase 1 LoRA...
✓ Model loaded


INPUT: EP at N2->N2,RP at N3->N1.

RAW OUTPUT:
Classify this CHIVA venous shunt.

INPUT: EP at N2->N2,RP at N3->N1.

TYPES:
TYPE 1: EP N1→N2, RP N2→N1, no EP N2→N3, no RP N3
TYPE 2A: EP N2→N3, no EP N1→N2
TYPE 2B: EP N2→N2, RP N3, no RP N2→N1, no EP N1→N2
TYPE 2C: EP N2→N2, RP N3, RP N2→N1, no EP N1→N2
TYPE 3: EP N1→N2, EP N2→N3, RP N3 only

Output: Type ____
Type of SHUNT is a classification system for the different types and subtypes (a,b,c)of varicose veins. It was developed by Dr.
Pier Luigi Zamboni in Italy to classify hemodynamic conditions that are found during clinical examination as well as duplex scanning,
and it has been widely accepted throughout Europe since its publication [6]. The main advantage over other systems such as Cockett or CEAP lies on how easy can be applied even if you have not any experience with ultrasound equipment because all information needed comes from physical exam without need additional tests like Valsalva ma

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

print("Testing Phase 1 Model...\n")

LORA_PATH = 'qwen_medical_lora_gpu'
BASE_PATH = 'Qwen25-7B'

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA
print("Loading Phase 1 LoRA...")
model = PeftModel.from_pretrained(
    base_model, 
    LORA_PATH,
    is_trainable=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✓ Model loaded\n")

# Test inference
print("="*80)
print("INFERENCE TEST WITH DECISION TABLE")
print("="*80)

DECISION_TABLE = """
QUICK DECISION TABLE:
Has EP N1→N2? YES + no EP N2→N3 + RP N2→N1 → TYPE 1
Has EP N1→N2? YES + EP N2→N3 + RP N3 only → TYPE 3
Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + elim="Reflux" → TYPE 1+2
No EP N1→N2 + EP N2→N3 → TYPE 2A
No EP N1→N2 + EP N2→N2 + RP N3 + NO RP N2→N1 → TYPE 2B
No EP N1→N2 + EP N2→N2 + RP N3 + RP N2→N1 → TYPE 2C
No EP N1→N2 + EP N2→N2 + NO RP → NO SHUNT
"""

test_clips = [
    "EP at N2->N2,RP at N3->N1.",
    "EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1.",
    "EP at N1->N2,RP at N2->N1."
]

for clip in test_clips:
    prompt = f"""FOLLOW THIS DECISION TABLE:
{DECISION_TABLE}

CLIP DATA: {clip}

STEP 1: Check if EP N1→N2 exists
STEP 2: Check if EP N2→N3 exists
STEP 3: Check RP patterns (N3, N2→N1)
STEP 4: Match to type using table above
STEP 5: Return ONLY JSON with shunt_classification and ligation_strategy keys

Do NOT repeat the input. Do NOT add extra text. Output ONLY JSON."""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=600,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=2.0,  # ADDED: Prevent repetition
            do_sample=True,
        )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\n{'='*80}")
    print(f"CLIP: {clip}")
    print(f"{'='*80}")
    print(f"OUTPUT:\n{result}\n")

print("="*80)
print("✓ Phase 1 model test complete")
print("="*80)

/home/krish/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Testing Phase 1 Model...

Loading base model...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 339/339 [00:04<00:00, 70.59it/s]


Loading Phase 1 LoRA...
✓ Model loaded

INFERENCE TEST WITH DECISION TABLE

CLIP: EP at N2->N2,RP at N3->N1.
OUTPUT:
FOLLOW THIS DECISION TABLE:

QUICK DECISION TABLE:
Has EP N1→N2? YES + no EP N2→N3 + RP N2→N1 → TYPE 1
Has EP N1→N2? YES + EP N2→N3 + RP N3 only → TYPE 3
Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + elim="Reflux" → TYPE 1+2
No EP N1→N2 + EP N2→N3 → TYPE 2A
No EP N1→N2 + EP N2→N2 + RP N3 + NO RP N2→N1 → TYPE 2B
No EP N1→N2 + EP N2→N2 + RP N3 + RP N2→N1 → TYPE 2C
No EP N1→N2 + EP N2→N2 + NO RP → NO SHUNT


CLIP DATA: EP at N2->N2,RP at N3->N1.

STEP 1: Check if EP N1→N2 exists
STEP 2: Check if EP N2→N3 exists
STEP 3: Check RP patterns (N3, N2→N1)
STEP 4: Match to type using table above
STEP 5: Return ONLY JSON with shunt_classification and ligation_strategy keys

Do NOT repeat the input. Do NOT add extra text. Output ONLY JSON. 
{
    "shunting_type": "",
	ligation_recommendations:""
} 

END WITH TWO NEWLINES AFTER OUTPUTTING THE RESULTS.
'''

import json
 
def classi

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

print("Testing Phase 1 Model...\n")

LORA_PATH = 'qwen_medical_lora_gpu'
BASE_PATH = 'Qwen25-7B'

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA - use absolute path without validation
print("Loading Phase 1 LoRA...")
model = PeftModel.from_pretrained(
    base_model, 
    LORA_PATH,
    is_trainable=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✓ Model loaded\n")

# Test inference
print("="*80)
print("INFERENCE TEST")
print("="*80)

test_prompts = [
    "Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N2->N2,RP at N3->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required.",
    "Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N1->N2,EP at N2->N3, RP at N3->N2, RP at N2->N1 . Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required.",
    "Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N1->N2,RP at N2->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required."
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=600, temperature=0.5)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {result}...\n")

print("="*80)
print("✓ Phase 1 model is working!")
print("="*80)

Testing Phase 1 Model...

Loading base model...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 339/339 [00:05<00:00, 64.97it/s]


Loading Phase 1 LoRA...
✓ Model loaded

INFERENCE TEST

Q: Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N2->N2,RP at N3->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required.
A: Given this Clip data , classify the shunt type (like for example : Type 1/Type 3/Type 2A/Type 2B/Type 2C/Type 1+2/ No Shunt) and give the appropriate ligation strategy : EP at N2->N2,RP at N3->N1. Instruction to follow : Do not repeat the input query and give output in the format of JSON with shunt_classification and ligation as the keys and your response as the values. do not add any extra unnecessary text before or after what is required. Make sure you have high confidence in your answe